# Session 0 — Python 101 for the course

Notebook of the Session 0 contact session, followed step by step in class (about 2–3 h on your own). Reference:
`course/session_00_python_101/guide.md` (installation, notebooks vs scripts, the libraries).

**How to work.** Run the cells in order (*Shift + Enter*). Before running a cell that says
**Predict**, write down what you expect. When the notebook is finished, *Restart* the kernel and
*Run All*: it must run from top to bottom without errors. That is the rule for every notebook in
the course.

**The data** is the weather of the case-study house: a typical meteorological year (TMY) for
Turin from PVGIS (European Commission, Joint Research Centre; see `data/processed/README.md`).

Contents: 0 check the environment · 1 Python essentials · 2 NumPy · 3 pandas time series · 4 Matplotlib · 5 SciPy ·
6 the course package · 7 what next.

## 0. Check the environment

The kernel must be the course environment (`.venv`). If the next cell fails with
`ModuleNotFoundError`, the wrong kernel is selected (guide §3).

In [ ]:
import sys

import matplotlib
import numpy as np
import pandas as pd
import scipy

import energy_training

print("Python      ", sys.version.split()[0])
print("NumPy       ", np.__version__)
print("pandas      ", pd.__version__)
print("Matplotlib  ", matplotlib.__version__)
print("SciPy       ", scipy.__version__)
print("course pkg  ", energy_training.__version__)

## 1. Python essentials

Only the parts of the language used in the course: numbers and strings, lists and
dictionaries, loops, functions, dataclasses, imports.

### 1.1 Numbers, names and units

Python does not know units. The course rule: **compute in SI** (temperatures in K, power in W,
energy in J) and convert to °C or kWh only to display. Put the unit in a comment or in the name. Symbols and names follow the course notation
(`docs/notation.md`): `T_amb` outdoor temperature, `Q_flow` heat flow rate, `UA` conductance.

In [ ]:
T_set_C = 20.0  # degC, indoor set point (display unit)
T_set = T_set_C + 273.15  # K, used in calculations
UA = 185.0  # W/K, heat-loss coefficient of the measured house used in Session 2 (example)

dT = 25.0  # K, indoor-outdoor difference on a cold day
Q_flow = UA * dT  # W, heat flow rate lost through the envelope
print(f"heat loss = {Q_flow:.0f} W = {Q_flow / 1000:.2f} kW")
n_days = 365
print(type(T_set), type(n_days), 7 / 2, 7 // 2, 2**10)

**f-strings** format numbers: `{x:.2f}` two decimals, `{x:.3g}` three significant digits,
`{x:,.0f}` thousands separator. Note `7 / 2` is `3.5` (true division) and `7 // 2` is `3`.

### 1.2 Lists, dictionaries, loops

A **list** is an ordered sequence; a **dict** maps keys to values.

In [ ]:
assets = ["heat pump", "thermal storage", "PV", "battery", "grid"]
carrier = {
    "heat pump": "electricity -> heat",
    "thermal storage": "heat",
    "PV": "solar -> electricity",
    "battery": "electricity",
    "grid": "electricity",
}

print(len(assets), assets[0], assets[-1], assets[1:3])
for name in assets:
    print(f"{name:16s} {carrier[name]}")

**Predict:** what does `[x**2 for x in range(4)]` return? A *list comprehension* builds a list
in one line. Run it.

In [ ]:
[x**2 for x in range(4)]

### 1.3 Functions

A function has a name, arguments, a docstring and a `return`. Type hints (`float`, `-> float`)
document what is expected; Python does not enforce them.

In [ ]:
def heat_loss(UA: float, T_set: float, T_amb: float) -> float:
    """Steady-state heat flow rate [W] lost by a house at T_set to the ambient at T_amb [K]."""
    return UA * (T_set - T_amb)


print(heat_loss(UA, T_set, 273.15 - 5.0))  # positional arguments
print(heat_loss(UA=UA, T_set=T_set, T_amb=273.15 + 25.0))  # keyword arguments: negative = gain
help(heat_loss)  # shows the docstring

### 1.4 Dataclasses: parameters with names

The course stores model parameters in **frozen dataclasses**: a small class whose fields have
names, types and defaults, and that cannot be changed after creation. Here: the site of the
case-study house, with the values PVGIS reports for the weather file.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Site:
    """Location of the case-study house."""

    name: str
    lat: float  # deg N
    lon: float  # deg E
    elevation: float  # m


turin = Site("Turin", lat=45.070, lon=7.687, elevation=250.0)
print(turin)
print(turin.lat)

try:
    turin.lat = 0.0
except Exception as err:  # frozen: a parameter set cannot be modified by mistake
    print(type(err).__name__, "-", err)

In Session 1 you will meet `ThermalZoneParams(C=..., UA=...)`, built the same way.

### 1.5 Imports and the course package

`import numpy as np` loads a module under a short name. The course package `energy_training`
is installed in the environment by `uv sync`, so it imports from any folder.

In [ ]:
from energy_training.utilities import celsius_to_kelvin, kelvin_to_celsius

print(celsius_to_kelvin(20.0), kelvin_to_celsius(293.15))
help(celsius_to_kelvin)

## 2. NumPy: arrays

A NumPy **array** holds numbers of one type and applies an operation to all of them at once
(*vectorisation*): no explicit loop, and much faster than a Python loop.

Example (illustrative values): an outdoor temperature over one winter day, sampled every hour.

In [ ]:
hours_of_day = np.arange(24)  # 0, 1, ..., 23
T_amb_day = celsius_to_kelvin(2.0 - 4.0 * np.cos(2 * np.pi * (hours_of_day - 3) / 24))  # K
print(hours_of_day.shape, T_amb_day.dtype)
print(np.round(T_amb_day[:6] - 273.15, 2))

**Predict:** `heat_loss` was written for single numbers. Does it work with an array?

In [ ]:
Q_flow_day = heat_loss(UA, T_set, T_amb_day)  # the same function, element by element
print(Q_flow_day.shape, f"min {Q_flow_day.min():.0f} W, max {Q_flow_day.max():.0f} W")

# energy over the day: each hourly value lasts 3600 s
E_day = np.sum(Q_flow_day * 3600.0)  # J
print(f"daily heat loss = {E_day / 3.6e6:.1f} kWh")

# the same with a Python loop (slower, longer, easier to get wrong)
E_loop = 0.0
for q in Q_flow_day:
    E_loop += q * 3600.0
print(np.isclose(E_day, E_loop))

**Boolean masks** select elements by a condition:

In [ ]:
cold = T_amb_day < celsius_to_kelvin(0.0)
print(cold.sum(), "hours below 0 degC, at hours", hours_of_day[cold])
print(f"mean loss in those hours: {Q_flow_day[cold].mean():.0f} W")

Two pitfalls to remember:

- `b = a` does **not** copy an array: both names point to the same data. Use `a.copy()`.
- Comparing floats with `==` is fragile: use `np.isclose` or a tolerance.

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = a
b[0] = 99.0
print(a)  # a changed too
print(0.1 + 0.2 == 0.3, np.isclose(0.1 + 0.2, 0.3))

## 3. pandas: time series

pandas adds **labels** to arrays: a `Series` is one labelled column, a `DataFrame` a table of
columns sharing one index. For time series the index holds time stamps.

### 3.1 Read the weather file

In [ ]:
from energy_training.utilities import repo_root

path = repo_root() / "data/processed/weather_turin_tmy.csv"
raw = pd.read_csv(path, parse_dates=["time_utc"])
raw.head()

In [ ]:
raw.info()

In [ ]:
raw.describe().round(1)

Columns (PVGIS names and units): `T2m` air temperature [°C], `RH` relative humidity [%],
`G(h)` global horizontal irradiance [W/m²], `Gb(n)` beam normal irradiance [W/m²],
`Gd(h)` diffuse horizontal irradiance [W/m²], `IR(h)` infrared irradiance [W/m²],
`WS10m` wind speed [m/s], `WD10m` wind direction [°], `SP` pressure [Pa].

### 3.2 Check the time stamps

**Predict:** consecutive rows are one hour apart, so every difference is 1 h. Check:

In [ ]:
steps = raw["time_utc"].diff()
steps.value_counts().head()

In [ ]:
jumps = raw.loc[steps != pd.Timedelta(hours=1), "time_utc"]
pd.DataFrame({"first stamp of the month": jumps.dt.strftime("%Y-%m-%d %H:%M")})

A **typical meteorological year** joins 12 months taken from different years (the most
"typical" January, the most typical February, ...). The time stamps jump between years.
Always look at the data before using it.

The course function `energy_training.data.weather.load_tmy()` does the cleaning once: it puts
the rows on one continuous hourly index (a nominal year, 2023, used as a label only), converts
the temperature to K and renames the columns. Its code is short: open
`python/src/energy_training/data/weather.py` and read it.

In [ ]:
from energy_training.data.weather import COLUMNS, load_tmy

tmy = load_tmy()
print({k: v[:2] for k, v in COLUMNS.items()})
tmy.head(3)

### 3.3 Select by time

With a time index, `.loc` accepts dates and date ranges.

In [ ]:
week = tmy.loc["2023-01-09":"2023-01-15"]
print(len(week), "hours")
week[["T_amb", "G_h"]].describe().round(1)

The stamps are in UTC. Local time in Turin is UTC+1 in winter and UTC+2 in summer;
`tz_convert` changes only the labels, not the data.

In [ ]:
tmy.index[4000], tmy.tz_convert("Europe/Rome").index[4000]

### 3.4 Resample and group

`resample` changes the time step (here hourly → daily, monthly); `groupby` collects rows with
the same key (here: the month number, or the hour of the day).

Irradiation (energy per m²) is approximated by treating each hourly irradiance value $G_h$ as
representative of its hour: $H_h \approx \sum G_h\,\Delta t$ with $\Delta t$ = 3600 s
(J/m², shown in kWh/m²). PVGIS labels every row HH:00, but the irradiance belongs to a slightly
different instant: the file header gives an *irradiance time offset* of 0.176 h (about 11 min)
for this site, which matters when the sun position is computed (Session 5), not for these sums.

`resample("D")` builds days from 00:00 to 24:00 **UTC**, one hour off local midnight in winter.

In [ ]:
J_PER_KWH = 3.6e6

daily_T = tmy["T_amb"].resample("D").mean()  # K
daily_H = tmy["G_h"].resample("D").sum() * 3600.0 / J_PER_KWH  # kWh/m2
print(len(daily_T), "days")
print(f"coldest day: {daily_T.idxmin():%d %b}, {daily_T.min() - 273.15:.1f} degC")

monthly_H = tmy["G_h"].resample("MS").sum() * 3600.0 / J_PER_KWH  # "MS": month start labels
print(monthly_H.round(1).to_string())
print(f"year: {monthly_H.sum():.0f} kWh/m2 on the horizontal plane")

In [ ]:
# the same monthly sums with groupby on the month number
by_month = (tmy["G_h"] * 3600.0 / J_PER_KWH).groupby(tmy.index.month).sum()
print(np.allclose(by_month.to_numpy(), monthly_H.to_numpy()))
tmy.groupby(tmy.index.month)["source_year"].first().to_dict()

## 4. Matplotlib: figures

The pattern used everywhere in the course: create a figure and axes with `plt.subplots`, call
methods on the axes, label every axis **with its unit**. `apply_style()` sets the course style;
`degc()` converts K → °C for display only.

In [ ]:
import matplotlib.pyplot as plt

from energy_training.plotting.style import apply_style, degc

apply_style()

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(7.0, 5.0))
ax1.plot(daily_T.index, degc(daily_T))
ax1.set(ylabel="daily mean T [°C]", title="Turin, typical meteorological year (PVGIS)")
ax2.bar(daily_H.index, daily_H, width=1.0)
ax2.set(ylabel="daily irradiation [kWh/m²]");

In [ ]:
fig, ax = plt.subplots()
ax.plot(week.index, degc(week["T_amb"]), label="air temperature")
ax.set(ylabel="T [°C]")
ax_r = ax.twinx()  # second y axis with its own unit
ax_r.plot(week.index, week["G_h"], color="C1", lw=1, label="global horizontal")
ax_r.set(ylabel="G [W/m²]")
ax.set_title("A winter week (UTC)")
fig.autofmt_xdate()
lines = ax.get_lines() + ax_r.get_lines()  # one legend for both axes
ax.legend(lines, [line.get_label() for line in lines], loc="upper left");

## 5. SciPy: a first fit

SciPy holds the numerical algorithms of the course: ODE solvers (`scipy.integrate`, from
Session 1), root finding and optimization (`scipy.optimize`, Session 8).

Here a short use of `scipy.optimize.curve_fit`: describe the annual temperature cycle with a
mean, an amplitude and the day of the coldest point,
$T(d) = \bar T - A\cos\!\left(2\pi\,(d - d_0)/365\right)$.

In [ ]:
from scipy.optimize import curve_fit


def annual_cycle(day, T_mean, A, d0):
    return T_mean - A * np.cos(2 * np.pi * (day - d0) / 365.0)


day = np.arange(len(daily_T), dtype=float)  # 0 = 1 January
popt, pcov = curve_fit(annual_cycle, day, daily_T.to_numpy(), p0=[285.0, 10.0, 15.0])
T_mean_fit, A_fit, d0_fit = popt
print(f"mean {T_mean_fit - 273.15:.1f} degC, amplitude {A_fit:.1f} K")
print(f"coldest point of the fitted cycle: day {d0_fit:.0f}")

fig, ax = plt.subplots()
ax.plot(day, degc(daily_T), lw=1, label="daily mean (TMY)")
ax.plot(day, degc(annual_cycle(day, *popt)), color="k", label="fitted cycle")
ax.set(xlabel="day of the year", ylabel="T [°C]")
ax.legend();

**Interpret.** The fitted cycle is coldest around mid-January, while the coldest single day
of this file is 6 February: the fit describes the average season, not one cold spell. The fit captures the season; the day-to-day weather is the scatter around it.
For energy questions the scatter matters too: cold spells set the heat-pump size, sunny and
cloudy days change the PV yield. That is why the course simulates with the hourly data.

## 6. A first look at the case study

A rough estimate of the yearly heat demand of the house: the steady-state loss
$\dot Q = UA\,(T_{set} - T_{amb})$ whenever $T_{amb} < T_{set}$, summed over the hours of the year.

$UA$ = 185 W/K is the value of the measured 120 m² house used in Session 2 (Bacher & Madsen,
2011), a house in Denmark: here it is only an example of the right order of magnitude.

> **Simplification:** no heat capacity, no solar or internal gains, constant set point, no hot
> water. Sessions 1–2 add the dynamics; Session 5 adds the full system. The number below is an
> upper-bound order of magnitude, not the house's demand.

In [ ]:
T_amb = tmy["T_amb"].to_numpy()  # K
Q_flow_heat = heat_loss(UA, T_set, T_amb)  # W, negative when warmer outside
Q_flow_heat = np.maximum(Q_flow_heat, 0.0)  # only heating
E_heat = Q_flow_heat.sum() * 3600.0 / J_PER_KWH
peak_time = tmy.index[Q_flow_heat.argmax()]
print(f"UA = {UA:.0f} W/K, set point {T_set_C:.0f} degC -> about {E_heat:,.0f} kWh/year")
print(f"peak loss: {Q_flow_heat.max() / 1000:.1f} kW at {peak_time:%d %b %H:%M} UTC")

Keep these two numbers in mind (energy per year, peak power): the first drives the running
cost, the second the size of the heat pump. The final question of the course is how to size
and operate the heat pump, storage, PV and battery to cover such a demand at minimum cost.

## 7. What next

1. Do the self-check exercise **S0.1**: `exercises/session_00/S0.1_python_self_check.md`.
2. Run the course tests in a terminal: `uv run pytest`. It must end with "passed".
3. *Restart* the kernel and *Run All* in this notebook: no errors.

Session 1 starts with a 10-minute check of these three points.